# OASIS production-tool exploration

This notebook inspects the same registered tools used by the formal pipeline. It does not provide runtime state to the CLI, FastAPI, or Agent. The default router is Haversine so running all cells does not consume ORS quota.

## 1. Environment and project root

In [ ]:
import sys
from pathlib import Path
from dotenv import load_dotenv

START_DIR = Path.cwd().resolve()
PROJECT_ROOT = next((path for path in (START_DIR, *START_DIR.parents)
                     if (path / "oasis").is_dir() and (path / "config").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"Cannot locate the OASIS project root from {START_DIR}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")
print("python:", sys.executable)
print("project root:", PROJECT_ROOT)

## 2. Confirmed task and isolated Context

Set `RUN_ORS = True` only when an ORS key is available and live routing is intended. The production safeguards and place-scoped cache still apply.

In [ ]:
from copy import deepcopy
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from oasis.agent.context import Context
from oasis.contracts.domain import TaskSpec
from oasis.services.registry import registry
from oasis.settings import get_weights, load_cfg

PLACE_NAME = "Los Angeles County, California, USA"
N_FACILITIES = 10
THRESHOLD_MINUTES = 15
RUN_ORS = False

cfg = deepcopy(load_cfg())
cfg["accessibility"]["router"] = "ors" if RUN_ORS else "haversine"
cfg["output"]["dir"] = str(PROJECT_ROOT / "outputs_run" / "notebook")

task = TaskSpec(
    place_name=PLACE_NAME,
    n_facilities=N_FACILITIES,
    threshold_minutes=THRESHOLD_MINUTES,
    weights=get_weights("balanced"),
)
ctx = Context()
ctx.put("config", cfg)
ctx.put("task", task)
print(task.model_dump())

In [ ]:
def run_tool(name, **kwargs):
    """Run one registered tool and surface any structured failure."""
    result = registry.get(name).fn(ctx=ctx, **kwargs)
    print(name, "ok=" + str(result.ok), result.summary)
    if not result.ok:
        raise RuntimeError(result.error.model_dump() if result.error else f"{name} failed")
    return result

## 3. Resolve the study area

The tool resolves the canonical boundary, FIPS codes, administrative level, and cache scope together. No FIPS value is hard-coded in downstream cells.

In [ ]:
run_tool("resolve_study_area", place_name=task.place_name)
study = ctx.get("study")
print({key: study[key] for key in ("place_name", "place_type", "state_fips", "county_fips", "county_fips_list", "cache_scope", "area_km2")})
study["boundary"].plot(figsize=(8, 8), facecolor="none", edgecolor="#185FA5")
plt.title(study["place_name"])
plt.axis("off")
plt.show()

## 4. Prepare Census and Landsat demand

`prepare_demand` performs the scoped ACS/TIGER/Landsat fetch, validates the join, imputes missing tract temperatures, and stores aligned demand objects in Context.

In [ ]:
run_tool("prepare_demand")
demand = ctx.get("demand")
areas = demand["areas"]
points = demand["points"]
population = demand["population"]
vulnerable_mask = demand["vulnerable_mask"]
indicators = demand["indicators"]

print("tracts:", len(areas))
print("CRS:", areas.crs)
print("heat quality:", ctx.get("quality")["heat"])
print(areas[["GEOID", "total_population", "elderly_ratio", "poverty_rate", "heat_exposure"]].head())

In [ ]:
assert len(areas) == len(points) == len(population)
assert areas["GEOID"].is_unique
assert areas.geometry.notna().all() and not areas.geometry.is_empty.any()
assert points.geometry.geom_type.eq("Point").all()
assert np.isfinite(population).all()
assert all(len(values) == len(areas) and np.isfinite(values).all() for values in indicators.values())
print("demand validation: ok")

### Older-adult population share

In [ ]:
bins = [0, .10, .15, .20, .25, np.inf]
labels = ["Below 10%", "10% to <15%", "15% to <20%", "20% to <25%", "25% or above"]
elderly_view = areas.copy()
elderly_view["elderly_group"] = pd.cut(elderly_view["elderly_ratio"], bins=bins, labels=labels, right=False, include_lowest=True)

fig, ax = plt.subplots(figsize=(12, 10))
elderly_view.plot(column="elderly_group", categorical=True, cmap="YlOrRd", legend=True,
                  edgecolor="white", linewidth=.15, missing_kwds={"color": "lightgrey", "label": "Missing data"},
                  legend_kwds={"title": "Population aged 65+", "loc": "lower left"}, ax=ax)
ax.set_title(f"Share of Population Aged 65+\n{study['place_name']}, {study['year']}")
ax.set_axis_off(); plt.tight_layout(); plt.show()

### Landsat land-surface temperature

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
areas.plot(column="heat_exposure", cmap="inferno", legend=True, edgecolor="white", linewidth=.15,
           legend_kwds={"label": "Mean land-surface temperature (C)"}, ax=ax)
ax.set_title(f"Median Summer Land-Surface Temperature\n{study['place_name']}")
ax.set_axis_off(); plt.tight_layout(); plt.show()

In [ ]:
heat_result = run_tool("summarize_heat_exposure", limit=10)
heat_summary = ctx.get("heat_summary")
print("heat artifacts:")
for name in ("map", "analysis_chart", "report"):
    print(" ", name, heat_summary[name])
print("spatial statistics:", heat_summary["statistics"])

## 5. Fetch candidate facilities

In [ ]:
run_tool("fetch_candidate_sites", amenities=task.candidate_amenities)
candidates = ctx.get("candidates")
print("candidate count:", len(candidates))
print(candidates[[column for column in ("cand_id", "name", "amenity") if column in candidates]].head())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
areas.boundary.plot(ax=ax, color="lightgrey", linewidth=.15)
candidates.to_crs(areas.crs).plot(ax=ax, color="#185FA5", markersize=8, alpha=.7, label="Candidate facilities")
ax.set_title(f"Candidate Cooling-Center Facilities\n{study['place_name']}")
ax.legend(); ax.set_axis_off(); plt.tight_layout(); plt.show()

## 6. Build the travel matrix

The registered tool chooses Haversine or guarded sparse ORS from `cfg`. ORS requests are place-scoped and reuse cached pairs.

In [ ]:
run_tool("build_travel_matrix")
travel = ctx.get("travel")
matrix = travel["matrix"]
print("router:", travel["router"])
print("shape:", matrix.shape)
print("diagnostics:", travel["diagnostics"])
assert matrix.shape == (len(areas), len(candidates))

In [ ]:
for threshold in (10, 15, 20, 30):
    reachable = (matrix <= threshold).any(axis=1)
    ratio = population[reachable].sum() / population.sum()
    print(f"{threshold:2d} min: {ratio:.2%} population reachable")

## 7. Optimize and inspect the Pareto frontier

In [ ]:
run_tool(
    "optimize_sites",
    n_facilities=task.n_facilities,
    threshold_minutes=task.threshold_minutes,
    weights=task.weights.model_dump(),
)
analysis = ctx.get("analysis")
solution = analysis["solution"]
frontier = analysis["frontier"]
print(solution.model_dump())
print("frontier points:", len(frontier))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter([point.efficiency_metric for point in analysis["points"]],
           [point.equity_metric for point in analysis["points"]], alpha=.35, label="Swept solutions")
ax.plot([point.efficiency_metric for point in frontier],
        [point.equity_metric for point in frontier], "o-", color="#EF9F27", label="Non-dominated frontier")
ax.set(xlabel="Total population coverage", ylabel="Vulnerable population coverage",
       title=f"Efficiency-Equity Frontier: N={task.n_facilities}, threshold={task.threshold_minutes:g} min")
ax.grid(alpha=.3); ax.legend(); plt.tight_layout(); plt.show()

## 8. Generate production-format outputs

In [ ]:
run_tool("generate_outputs")
outputs = ctx.get("outputs")
for name, output_path in outputs.items():
    print(name, output_path, "exists=" + str(Path(output_path).is_file()))
assert all(Path(output_path).is_file() for output_path in outputs.values())

The fixed CLI pipeline calls the same six registered tools in this order. Agent mode adds typed parsing, observation-driven tool selection, bounded correction, HITL, and evidence-grounded interpretation around these deterministic operations.